# Feature Engineering

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline   
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_squared_error
from sklearn.impute import SimpleImputer
from sklearn.feature_selection import SelectKBest, f_regression, RFE
from sklearn.linear_model import LinearRegression
from sklearn.cluster import KMeans

from xgboost import XGBRegressor

In [2]:
data_original = pd.read_parquet(".cache/vehicles_cleaned.parquet")
data = data_original.copy()

## 'Age' feature

In [3]:
data['year'].max()

np.float64(2022.0)

In [4]:
# Create 'age' feature from 'year' column

def create_age_feature(df):
    df['age'] = 2022 - df['year']
    
    return df

## Lat/Long Clusters

In [5]:
# Create clusters from 'lat' and 'long' columns

def create_location_clusters(df, n_clusters):
    kmeans = KMeans(n_clusters=n_clusters, random_state=42)
    
    df['location_cluster'] = kmeans.fit_predict(df[['lat', 'long']])
    
    return df

## Rare categories grouping

In [6]:
# Group rare categories in a categorical column, replacing them with 'Other' if their frequency is below the threshold,
# for manufacturer, condition, cylinders, gas, transmission, fuel, type

def group_rare_categories(df, column, threshold=0.01):
    
    value_counts = df[column].value_counts(normalize=True)
    rare_categories = value_counts[value_counts < threshold].index
    df[column] = df[column].apply(lambda x: 'Other' if x in rare_categories else x)
    
    return df

## 'Model' Encoding

In [7]:
# Add a new column 'model_freq' representing the frequency of each model and drop the original 'model' column,
# The 'model' column will be replaced with its frequency count.

def add_model_frequency(data):

    freq = data['model'].value_counts()
    data['model'] = data['model'].map(freq)
        
    return data

In [8]:
class FeatureEngineering:
    def __init__(self, data):
        self.data = data

    def create_location_clusters(self, n_clusters):
        kmeans = KMeans(n_clusters=n_clusters, random_state=42)
        self.data['location_cluster'] = kmeans.fit_predict(self.data[['lat', 'long']])
        return self.data
    
    def create_age_feature(self):
        self.data['age'] = 2024 - self.data['year']
        return self.data

    def create_mileage_per_year_feature(self):
        self.data['mileage_per_year'] = self.data['odometer'] / self.data['age']
        return self.data
    
    def add_model_frequency(self):
        model_counts = self.data['model'].value_counts()
        self.data['model_frequency'] = self.data['model'].map(model_counts)
        return self.data
    
    def rare_categories_grouping(self, column, threshold=0.01):
        value_counts = self.data[column].value_counts(normalize=True)
        rare_categories = value_counts[value_counts < threshold].index
        self.data[column] = self.data[column].replace(rare_categories, 'Other')
        return self.data
    
    # Make a final method to get the processed data
    def get_data(self):
        return self.data

In [9]:
def drop_region_column(data):
    return data.drop(columns=['region'])

## Evaluation with engineered features

In [10]:
from sklearn.model_selection import KFold, cross_validate
from sklearn.compose import make_column_selector
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_squared_error, r2_score, mean_absolute_error
from car_pipeline import OutlierDetector, FeatureEngineeringTransformer

In [11]:
# Same prep as the baseline, but keep 'model' (needed for model_frequency; the transformer drops it)
data = data_original.copy()
data = data.drop(columns=['region']).dropna(subset=['price'])

X = data.drop(columns=['price'])
y = data['price']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

detector = OutlierDetector().fit(X_train, y_train)
X_train, y_train = detector.transform(X_train, y_train)
X_test, y_test = detector.transform(X_test, y_test, filter_price=True, filter_odometer=False)

In [12]:
def get_model(model_name):
    if model_name == 'linear_regression':
        return LinearRegression()
    elif model_name == 'random_forest':
        return RandomForestRegressor(n_estimators=300, random_state=42, n_jobs=-1, max_depth=5)
    elif model_name == 'gradient_boosting':
        return XGBRegressor(n_estimators=300, random_state=42, n_jobs=-1, learning_rate=0.05)
    raise ValueError(f"Model {model_name} not recognized")


def build_pipeline(model_name, n_clusters=50, drop_columns=()):
    numeric_transformer = Pipeline(steps=[
        ('imputer', SimpleImputer(strategy='mean')),
        ('scaler', StandardScaler())
    ])
    categorical_transformer = Pipeline(steps=[
        ('imputer', SimpleImputer(strategy='most_frequent')),
        ('encoder', OneHotEncoder(handle_unknown='ignore', drop='first'))
    ])
    # Columns are picked by dtype because the engineered columns only exist after the first step
    preprocessor = ColumnTransformer([
        ('num', numeric_transformer, make_column_selector(dtype_include=np.number)),
        ('cat', categorical_transformer, make_column_selector(dtype_exclude=np.number))
    ])
    return Pipeline([
        ('features', FeatureEngineeringTransformer(n_clusters=n_clusters, drop_columns=drop_columns)),
        ('preprocessor', preprocessor),
        ('model', get_model(model_name))
    ])


def train_and_evaluate_model(model_name, X_train, y_train, n_splits=3, drop_columns=()):
    cv = KFold(n_splits=n_splits, shuffle=True, random_state=42)
    scores = cross_validate(
        build_pipeline(model_name, drop_columns=drop_columns), X_train, y_train, cv=cv,
        scoring={'mse': 'neg_mean_squared_error', 'mae': 'neg_mean_absolute_error', 'r2': 'r2'}
    )
    return {
        'model': model_name,
        'mse': -scores['test_mse'].mean(), 'mse_std': scores['test_mse'].std(),
        'mae': -scores['test_mae'].mean(), 'mae_std': scores['test_mae'].std(),
        'r2': scores['test_r2'].mean(), 'r2_std': scores['test_r2'].std(),
    }


def final_test_evaluation(model_name, X_train, y_train, X_test, y_test, drop_columns=()):
    # Run once, after model/feature choices are final
    pipe = build_pipeline(model_name, drop_columns=drop_columns).fit(X_train, y_train)
    y_pred = pipe.predict(X_test)
    mse = mean_squared_error(y_test, y_pred)
    return {'model': model_name, 'mse': mse, 'rmse': mse ** 0.5,
            'mae': mean_absolute_error(y_test, y_pred), 'r2': r2_score(y_test, y_pred)}


from pathlib import Path
import joblib

RESULTS_DIR = Path(".cache/engineered")
RESULTS_DIR.mkdir(parents=True, exist_ok=True)


def cached_eval(model_name, X_train, y_train, force=False):
    # Delete the file (or pass force=True) after changing features, folds or model settings
    path = RESULTS_DIR / f"{model_name}.joblib"
    if path.exists() and not force:
        return joblib.load(path)
    result = train_and_evaluate_model(model_name, X_train, y_train)
    joblib.dump(result, path)
    return result


In [13]:
results_linear_regression = cached_eval('linear_regression', X_train, y_train)
results_linear_regression

{'model': 'linear_regression',
 'mse': np.float64(58936104.91515928),
 'mse_std': np.float64(304392.61811688385),
 'mae': np.float64(5412.3049623135885),
 'mae_std': np.float64(26.457710542002047),
 'r2': np.float64(0.688747279905091),
 'r2_std': np.float64(0.0015035858508800266)}

In [14]:
results_gradient_boosting = cached_eval('gradient_boosting', X_train, y_train)
results_gradient_boosting

{'model': 'gradient_boosting',
 'mse': np.float64(35002557.333333336),
 'mse_std': np.float64(202532.12513135347),
 'mae': np.float64(3787.5902506510415),
 'mae_std': np.float64(9.542250442554089),
 'r2': np.float64(0.8151444395383199),
 'r2_std': np.float64(0.0010882986004046118)}

In [15]:
results_random_forest = cached_eval('random_forest', X_train, y_train)
results_random_forest

{'model': 'random_forest',
 'mse': np.float64(70419062.53519948),
 'mse_std': np.float64(522894.1144399734),
 'mae': np.float64(5919.3487066054395),
 'mae_std': np.float64(23.22197039791496),
 'r2': np.float64(0.6281028483854975),
 'r2_std': np.float64(0.00278059932694859)}

## Ablation: leave one feature out

In [16]:
def ablation(model_name, X_train, y_train, features, n_splits=3):
    # Re-runs the CV with each feature removed; delta > 0 in r2 means the model got better without it
    base = train_and_evaluate_model(model_name, X_train, y_train, n_splits)
    rows = [{'dropped': '(none)', 'r2': base['r2'], 'r2_std': base['r2_std'], 'mae': base['mae'],
             'd_r2': 0.0, 'd_mae': 0.0}]
    for feat in features:
        res = train_and_evaluate_model(model_name, X_train, y_train, n_splits, drop_columns=[feat])
        rows.append({'dropped': feat, 'r2': res['r2'], 'r2_std': res['r2_std'], 'mae': res['mae'],
                     'd_r2': res['r2'] - base['r2'], 'd_mae': res['mae'] - base['mae']})
    return pd.DataFrame(rows).sort_values('d_r2', ascending=False).reset_index(drop=True)


ablation_features = [
    # engineered
    'age', 'mileage_per_year', 'location_cluster', 'model_frequency',
    # raw
    'year', 'odometer', 'lat', 'long', 'manufacturer', 'condition', 'cylinders', 'fuel',
    'title_status', 'transmission', 'drive', 'size', 'type', 'paint_color', 'state',
]

In [18]:
#ablation_gb = ablation('gradient_boosting', X_train, y_train, ablation_features)
#ablation_gb

In [19]:
ablation_lr = ablation('linear_regression', X_train, y_train, ablation_features)
ablation_lr

c:\Users\Luka\Documents\Projects\car-leasing\.venv\Lib\site-packages\sklearn\preprocessing\_encoders.py:262: UserWarning: Found unknown categories in columns [0] during transform. These unknown categories will be encoded as all zeros
  warnings.warn(msg, UserWarning)
c:\Users\Luka\Documents\Projects\car-leasing\.venv\Lib\site-packages\sklearn\preprocessing\_encoders.py:262: UserWarning: Found unknown categories in columns [0] during transform. These unknown categories will be encoded as all zeros
  warnings.warn(msg, UserWarning)
c:\Users\Luka\Documents\Projects\car-leasing\.venv\Lib\site-packages\sklearn\preprocessing\_encoders.py:262: UserWarning: Found unknown categories in columns [0] during transform. These unknown categories will be encoded as all zeros
  warnings.warn(msg, UserWarning)
c:\Users\Luka\Documents\Projects\car-leasing\.venv\Lib\site-packages\sklearn\preprocessing\_encoders.py:262: UserWarning: Found unknown categories in columns [0] during transform. These unknown ca

,dropped,r2,r2_std,mae,d_r2,d_mae
0,model_frequency,0.688750,0.001492,5413.206687,0.000003,0.901725
1,(none),0.688747,0.001504,5412.304962,0.000000,0.000000
2,year,0.688645,0.001446,5412.507662,-0.000102,0.202700
3,lat,0.688635,0.001495,5413.069050,-0.000112,0.764088
4,long,0.688634,0.001443,5413.315097,-0.000114,1.010135
5,age,0.688626,0.001439,5412.561383,-0.000122,0.256421
6,mileage_per_year,0.688607,0.001470,5415.263581,-0.000140,2.958619
7,size,0.688500,0.001497,5415.042201,-0.000248,2.737239
8,condition,0.688288,0.001514,5418.417723,-0.000459,6.112761
9,paint_color,0.688188,0.001493,5413.545218,-0.000559,1.240255


## XGBoost with selected features

In [ ]:
DROP_COLUMNS = ['state', 'size', 'year', 'location_cluster']

selected_path = RESULTS_DIR / "gradient_boosting_selected.joblib"
if selected_path.exists():
    results_selected = joblib.load(selected_path)
else:
    results_selected = train_and_evaluate_model('gradient_boosting', X_train, y_train, drop_columns=DROP_COLUMNS)
    joblib.dump(results_selected, selected_path)

results_selected

c:\Users\Luka\Documents\Projects\car-leasing\.venv\Lib\site-packages\sklearn\preprocessing\_encoders.py:262: UserWarning: Found unknown categories in columns [0] during transform. These unknown categories will be encoded as all zeros
  warnings.warn(msg, UserWarning)


{'model': 'gradient_boosting',
 'mse': np.float64(34911989.333333336),
 'mse_std': np.float64(154435.4680275515),
 'mae': np.float64(3764.9239908854165),
 'mae_std': np.float64(19.394645446598222),
 'r2': np.float64(0.8156231244405111),
 'r2_std': np.float64(0.0007539700644851152)}

In [ ]:
final_test_evaluation('gradient_boosting', X_train, y_train, X_test, y_test, drop_columns=DROP_COLUMNS)

{'model': 'gradient_boosting',
 'mse': 35979496.0,
 'rmse': 5998.291089968876,
 'mae': 3804.0693359375,
 'r2': 0.8113240003585815}

## Linear Regression with selected features

In [ ]:
final_test_evaluation('linear_regression', X_train, y_train, X_test, y_test, drop_columns=DROP_COLUMNS)